# Hello EIQ — zero to your first Himalayas submission

From `pip install` to a real, model-backed submission, built around EIQ's own SDK.

> **Closed beta.** The tournament runs on **staging**, behind **Cloudflare Access**. You authenticate with an API key **and** a Cloudflare Access service token (both come from onboarding's *Copy setup command*). When EIQ opens publicly, drop the token and point at the public site.

**This notebook walks through:**
1. Authenticate
2. Explore the round & dataset schema
3. Download the data (train / validation / live)
4. Understand the data — feature sets, expeds, the target, feature bins
5. Train a LightGBM model on `target_everest_20`
6. Evaluate it — data leakage, the embargo, per-exped CORR, Sharpe, drawdown, and AIMC
7. Predict the live round and (optionally) submit
8. Read your scores

**Payout:** `0.75 * CORR + 2.25 * AIMC`, scored over a 20-day forward window. AIMC — your alpha over the stake-weighted ai-model consensus — is weighted **3x CORR**, so differentiated predictions win.

## 1. Install

In [ ]:
# everestapi is the EIQ SDK; the rest are for training, evaluation, and plotting.
%pip install --quiet "everestapi>=0.2.1" lightgbm pandas pyarrow scikit-learn scipy matplotlib

## 2. Authenticate (closed beta)

Credentials come from onboarding's **Copy setup command** (*Install the SDK & submit your first prediction -> Step 2*). Set them in your shell **before** launching Jupyter (or in a local `.env`) — never commit them:

`EIQ_API_KEY`, `EIQ_BASE_URL`, `CF_ACCESS_CLIENT_ID`, `CF_ACCESS_CLIENT_SECRET`

In [ ]:
import os
from everestapi import EverestAPI

# Read every credential from the environment, so nothing secret is written into the notebook.
base_url = os.environ.get("EIQ_BASE_URL", "https://staging.everesteer.ai")
api_key = os.environ.get("EIQ_API_KEY") or os.environ.get("EVEREST_API_KEY")
cf_id = os.environ.get("CF_ACCESS_CLIENT_ID")
cf_secret = os.environ.get("CF_ACCESS_CLIENT_SECRET")

# Fail fast with a clear message instead of a cryptic 401/403 deeper in the notebook.
if not api_key:
    raise RuntimeError("Set EIQ_API_KEY (onboarding -> Copy setup command).")
if "staging" in base_url and not (cf_id and cf_secret):
    raise RuntimeError(
        "Closed beta: staging is behind Cloudflare Access. Set CF_ACCESS_CLIENT_ID "
        "and CF_ACCESS_CLIENT_SECRET from onboarding's Copy setup command."
    )

client = EverestAPI(
    api_key=api_key,
    base_url=base_url,
    cf_access_client_id=cf_id,
    cf_access_client_secret=cf_secret,
)
client.health()  # returns {'status': 'ok', ...} on a working, authenticated connection

## 3. Explore the round & dataset schema

`get_current_round` shows the open round and its timing; `get_dataset_schema` is the map of the data — the feature sets, feature groups, and the targets you can train on.

In [ ]:
# The current round tells you what's open and the key dates (close / score / resolve).
round_info = client.get_current_round(tournament="futures")
print("Round:", round_info["round_number"], "| status:", round_info["status"])
print("Universe size:", round_info["universe_size"])
print("Closes:", round_info["close_at"], "| resolves:", round_info["resolve_at"])

# The schema is the map of the data: which feature sets/groups and targets exist.
schema = client.get_dataset_schema()
feature_sets = schema["feature_sets"]
# `targets` may arrive as a list of names or a name->definition dict; normalize to a list.
target_names = list(schema["targets"])
for s in ["small", "medium", "all"]:
    print(f"feature set '{s}':", len(feature_sets[s]), "features")
print("targets:", len(target_names), "->", target_names[:4], "...")

PRIMARY_TARGET = "target_everest_20"   # the scored / payout target (20-day forward return)
print("Primary target:", PRIMARY_TARGET)

## 4. Download the data

`download_dataset` writes a parquet and returns its path; the version resolves automatically.
- **train** — features + targets, history (fit on this)
- **validation** — features + targets, with a `data_type` column (evaluate on this)
- **live** — current-round features only (predict on this)

In [ ]:
import pandas as pd

# download_dataset writes a parquet locally and returns its path; the version auto-resolves.
train = pd.read_parquet(client.download_dataset("futures", "train"))
val = pd.read_parquet(client.download_dataset("futures", "validation"))
live = pd.read_parquet(client.download_dataset("futures", "live"))

EXPED = "exped"   # each exped = one trading day (an "expedition")

# Start with the 'small' set (fewer features = faster, less memory); scale up later for more signal.
feature_set = feature_sets["small"]

def exped_num(e):
    # Expeds are strings like "exped_8978"; pull the integer so we can order them in time.
    return int(str(e).split("_")[-1])

# Downsample to every 4th exped purely for speed in this walkthrough; use all of it for real models.
keep = sorted(train[EXPED].unique(), key=exped_num)[::4]
train = train[train[EXPED].isin(keep)]

print(f"train {train.shape} | val {val.shape} | live {live.shape}")
print(f"features used: {len(feature_set)} (of {len(feature_sets['all'])})")

## 5. Understand the data

The Himalayas dataset is **tabular and fully obfuscated** — instrument ids, feature names, and target definitions are anonymized, so you can model it without financial domain knowledge or bias. Every row is **one instrument on one `exped`** (a single trading day), and the columns fall into a few groups:

- **`id`** (the index, e.g. `eiq_504084dfb312a97e`) — the instrument
- **`exped`** — the trading day; all rows sharing an exped are one snapshot of the market
- **features** (`feature_*`) — quintile-binned attributes of each instrument
- **targets** (`target_*`) — the forward returns you predict

Two extra columns ride along: `data_type` (which split a row belongs to) and `climb_difficulty` (a static 1-5 rating, constant per instrument).

In [ ]:
# Group the columns so the shape of the data is obvious at a glance.
meta_cols = [c for c in ["exped", "data_type", "climb_difficulty"] if c in train.columns]
target_cols = list(schema["targets"])   # normalized list of target names (see step 3)
all_feature_cols = [c for c in train.columns if c.startswith("feature_")]

print(f"{train.shape[0]:,} rows x {train.shape[1]:,} columns")
print(f"  expeds:   {train[EXPED].nunique():>5}  (each = one trading day)")
print(f"  features: {len(all_feature_cols):>5}  (using {len(feature_set)} from the 'small' set)")
print(f"  targets:  {len(target_cols):>5}")
print(f"  metadata: id (index) + {meta_cols}")

# A peek at the table: one exped, a few features, and the primary target.
train[[EXPED] + feature_set[:4] + [PRIMARY_TARGET]].head()

### Expeds

Each `exped` is one trading day; the rows within it are the investable universe that day. Scoring is **per exped**, so it helps to treat an exped as a single example. Here is how many instruments appear each exped:

In [ ]:
import matplotlib.pyplot as plt

# The universe drifts over time, so the row count per exped is not constant. Order by exped
# number (not the default lexicographic string sort) so the x-axis runs in true time order.
counts = train.groupby(EXPED).size()
counts = counts.reindex(sorted(counts.index, key=exped_num))
counts.plot(title="Instruments per exped", figsize=(8, 3), xlabel="exped", ylabel="rows", xticks=[])
plt.tight_layout(); plt.show()

### Features

Features are **quantitative attributes** of each instrument, binned into **5 integer levels `0, 1, 2, 3, 4`** (heavy binning regularizes noisy underlying values). A value of **`-1` means the feature was missing** for that row. Names are obfuscated but descriptive, e.g. `feature_grooved_couloir_diluted`.

In [ ]:
# Features are integer bins 0..4; -1 is the missing-data sentinel (not a real bin).
example_feat = feature_set[0]
print("Example feature:", example_feat)
train[example_feat].value_counts(dropna=False).sort_index().plot(
    kind="bar", title=f"Value distribution: {example_feat}",
    figsize=(5, 3), xlabel="bin  (-1 = missing)", ylabel="rows"
)
plt.tight_layout(); plt.show()

### Feature groups

Beyond the `small`/`medium`/`all` size tiers, features are organized into **named groups** (`glacier`, `ridge`, `summit`, `valley`, ...), each a different *kind* of signal. Groups matter later for feature-exposure analysis and neutralization. Below: each group's size and how much of it the `small` set includes.

In [ ]:
# Each feature belongs to a named group. The served schema carries the groups alongside the
# size tiers in `feature_sets`; some schema versions expose them under `feature_groups`
# instead — handle both so this never silently produces an empty table.
size_tiers = {"small", "medium", "all"}
group_members = {g: m for g, m in feature_sets.items() if g not in size_tiers}
if not group_members and "feature_groups" in schema:
    # feature_groups maps name -> {"features": [...], ...} or name -> [...]
    group_members = {
        g: (v["features"] if isinstance(v, dict) else v)
        for g, v in schema["feature_groups"].items()
        if g not in size_tiers
    }

small_set = set(feature_sets["small"])
group_table = pd.DataFrame({
    "group_size": {g: len(m) for g, m in group_members.items()},
    "in_small":   {g: len(set(m) & small_set) for g, m in group_members.items()},
}).sort_values("group_size", ascending=False)
group_table

### Targets

What you predict is a **forward return**. There are **16 targets** = 8 "peaks" (`everest`, `k2`, `lhotse`, `makalu`, `manaslu`, `nanga_parbat`, `nuptse`, `gyachung_kang`) x 2 horizons (the `_20` / `_60` suffix = trading days ahead). Each peak is a *different definition* of forward return.

The **scored / payout target is `target_everest_20`** (a 20-day forward return), binned into 5 levels — `0, 0.25, 0.5, 0.75, 1.0`. The other 15 are **auxiliary targets**: not scored, but useful to train on and ensemble — and in EIQ they are all fully populated, so there are no missing target values to handle.

> One pair of peaks — `manaslu` and `nanga_parbat` — are exact inverses of each other (correlation ≈ −1.0). When ensembling auxiliary targets, keep only one of the two.

In [ ]:
# All targets share the same 5-level binning; here is the scored one.
train[PRIMARY_TARGET].value_counts().sort_index().plot(
    kind="bar", title=f"{PRIMARY_TARGET} distribution (5 bins)",
    figsize=(5, 3), xlabel="target value", ylabel="rows"
)
plt.tight_layout(); plt.show()

In [ ]:
# Targets that correlate strongly with everest_20 capture similar returns; weakly-correlated
# ones add diversity and make the more interesting candidates for an ensemble. Watch for a
# pair sitting at corr ~ -1.0 (manaslu / nanga_parbat) — they are inverses, so keep only one.
(train[target_cols].corrwith(train[PRIMARY_TARGET])
    .sort_values(ascending=False)
    .to_frame("corr_with_everest_20"))

## 6. Train a model

Your job is to predict `target_everest_20` from the features. Any model works — here are the main families people reach for on tabular data like this:

- **Tree-based models** — the usual default for noisy, low-signal tabular data. They handle the binned integer features and missing values gracefully and capture non-linear interactions without feature scaling.
  - *Random Forest* — bagged trees; a robust, low-tuning baseline.
  - *XGBoost* / *LightGBM* — gradient-boosted trees; strong accuracy, fast, the most popular choice in tournaments like this.
  - *CatBoost* — gradient boosting with first-class **categorical** handling, which suits these quintile-binned features.
- **Linear models** — *Ridge* / *Lasso* / *ElasticNet*. Fast, low-variance baselines that rarely overfit; they miss feature interactions but are excellent for ensembling and as a neutralization base.
- **Neural networks** — MLPs / tabular nets. Can model rich interactions, but need careful regularization and feature scaling on this low signal-to-noise data; most often used inside an ensemble rather than alone.

(EIQ's serverless `quick_train` supports `lightgbm`, `xgboost`, `ridge`, and `mlp` out of the box if you'd rather not train locally.)

Below we use **LightGBM** — a gradient-boosted tree model that's a strong, fast default. We map the `-1` missing sentinel to `NaN` so LightGBM treats it as missing rather than as a real bin.

In [ ]:
import lightgbm as lgb

def features_matrix(df, cols):
    # Cast to float and turn the -1 missing sentinel into NaN, so LightGBM treats it as
    # "missing" (it handles NaN natively) instead of as a real bin between 0 and 4.
    X = df[cols].astype("float32")
    return X.where(X >= 0)

# Lightly-regularized starting params (not tuned). The two comments below are the knobs
# that matter most on this noisy, low-signal data.
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=6,
    num_leaves=64,
    colsample_bytree=0.1,    # few features per tree -> more diverse trees, less overfit
    min_child_samples=500,   # large leaves -> smoother predictions, less overfit
    random_state=42,
    verbose=-1,
)
model.fit(features_matrix(train, feature_set), train[PRIMARY_TARGET])
print("Trained on", len(train), "rows.")

## 7. Evaluate

Before trusting a model, evaluate it **out-of-sample** on the validation split. The tournament scores **per exped**, so we measure correlation within each exped, then summarize across expeds.

### Data leakage and the embargo

`target_everest_20` is a **20-day forward return** — each target value already encodes what happens over the next ~20 trading days. So the **last expeds of training overlap in time with the first expeds of validation**: a model evaluated on those overlapping expeds would be partly judged on information that bled in from training. That optimistic bias is **data leakage**, and it makes a backtest look better than live performance.

The fix is an **embargo**: drop the validation expeds that fall within the target's 20-day window after the last training exped, so train and validation never share an overlapping forward-return period. Since one exped is one trading day, dropping the full **20 expeds** guarantees zero overlap. (The platform's exped-purged cross-validation uses a smaller embargo for its internal splits; here we drop the entire forward horizon to be unambiguously leak-free.) Skipping this is one of the most common ways people fool themselves with good-looking validation scores.

In [ ]:
# Evaluate only on rows that actually have targets (data_type == "validation").
val_eval = val[val["data_type"] == "validation"].copy()

# Embargo: the target looks 20 trading days ahead and one exped == one trading day, so a
# validation exped within 20 of the last training exped shares part of its forward-return
# window and would leak. Dropping the full 20-exped horizon guarantees zero overlap.
EMBARGO = 20
last_train_exped = max(train[EXPED].map(exped_num))
val_eval = val_eval[val_eval[EXPED].map(exped_num) > last_train_exped + EMBARGO]
print(f"Embargo {EMBARGO} expeds | validation expeds kept: {val_eval[EXPED].nunique()}")

### Per-exped CORR, Sharpe, and drawdown

CORR is the rank correlation between your predictions and the target, computed **within each exped** and then summarized. We also report **Sharpe** (consistency = mean / std of per-exped CORR) and **max drawdown** (worst peak-to-trough of the cumulative score) — the same risk lens the tournament uses.

In [ ]:
import numpy as np
from scipy.stats import spearmanr

# Predict on the embargoed validation rows.
val_eval["prediction"] = model.predict(features_matrix(val_eval, feature_set))

def per_exped_corr(df):
    # Spearman (rank) correlation between prediction and target, one value per exped.
    out = {}
    for e, g in df.groupby(EXPED):
        rho, _ = spearmanr(g["prediction"], g[PRIMARY_TARGET])
        if np.isfinite(rho):
            out[e] = rho
    # Order by exped number (not the default lexicographic key) so the cumulative curve and
    # the max-drawdown below are computed in true chronological order.
    s = pd.Series(out)
    return s.reindex(sorted(s.index, key=exped_num))

corr = per_exped_corr(val_eval)
# mean = average edge per exped; std = volatility; Sharpe = consistency; % positive = hit rate.
print(f"Mean CORR {corr.mean():.4f} | Std {corr.std():.4f} | "
      f"Sharpe {corr.mean() / corr.std():.2f} | % positive {(corr > 0).mean() * 100:.0f}%")
# Max drawdown: largest drop from a running peak of the (chronological) cumulative CORR curve.
dd = (corr.cumsum().expanding().max() - corr.cumsum()).max()
print(f"Max drawdown {dd:.4f}")

# Cumulative CORR is a rough "backtest" of the model over the validation period.
corr.cumsum().plot(title="Cumulative validation CORR", figsize=(8, 3), xticks=[])
plt.tight_layout(); plt.show()

### Where AIMC comes from

CORR you can measure offline (above). **AIMC** — your alpha over the stake-weighted ai-model consensus, the 3x-weighted payout term — is computed **server-side**. Two ways to see it:

- After you submit and the round scores: `client.get_scores(model_id=...)` returns per-day CORR, AIMC, and payout, and `client.get_validation_diagnostics(model_id=...)` returns the fuller panel (per-exped CORR / EAC, Sharpe, drawdown).
- Offline, download the consensus to compare your predictions against it:

In [ ]:
# The ai-model consensus is version-scoped, so resolve the current version first.
versions = client.list_versions().get("versions", [])
ver = next((v["version"] for v in versions if "futures" in (v.get("universes") or [])),
           (versions[0]["version"] if versions else "v0"))
try:
    # These are the stake-weighted consensus predictions your AIMC is measured against.
    ai_model = pd.read_parquet(client.download_ai_model(universe="futures", version=ver))
    print("Consensus (ai-model) predictions:", ai_model.shape)
    display(ai_model.head())
except Exception as e:
    print("Consensus download not available in this round:", e)

## 8. Predict the live round & submit

Build a `{instrument_id: prediction}` dict over the live universe, then submit with `submit_futures_predictions` (the futures call). Your model must exist first (`create_model`). Submission is **off by default** — set `SUBMIT = True` to actually submit.

In [ ]:
SUBMIT = False                      # safety switch: keep False to avoid touching the live round
MODEL_ID = "hello-eiq-baseline"

# Predict the live round and shape it as {instrument_id: prediction} over the live universe.
live_preds = model.predict(features_matrix(live, feature_set))
# Instrument ids live in the `id` column if present, otherwise in the index — handle both so
# the submission keys are always real instrument ids, never a positional 0, 1, 2, ... range.
live_ids = live["id"] if "id" in live.columns else live.index
predictions = {str(i): float(p) for i, p in zip(live_ids, live_preds)}
print(f"{len(predictions)} live predictions ready. Example id: {next(iter(predictions))}")

if SUBMIT:
    # A model must exist before you can submit to it (create_model is idempotent).
    client.create_model(MODEL_ID, description="hello_eiq LightGBM baseline")
    # Futures uses submit_futures_predictions (NOT submit_predictions, which is the equities call).
    result = client.submit_futures_predictions(model_id=MODEL_ID, predictions=predictions)
    print("Submitted:", result)
else:
    print("Dry run — set SUBMIT = True to submit.")

## 9. Check your scores

After the round scores (T+1 onward), read your CORR / AIMC / EAC / payout. Empty until the first partials land.

In [ ]:
# Scores accrue over the T+1..T+20 window; empty until the first partials land — and the
# model must have been submitted at least once, so this can 404 on a dry run. Handle both.
try:
    scores = client.get_scores(model_id=MODEL_ID, days=30)
    df_scores = pd.json_normalize(scores.get("scores", []))
    if df_scores.empty:
        print("No partials yet — scores land from T+1 once the round starts scoring.")
except Exception as e:
    print("No scores available yet (submit first, then re-run after T+1):", e)
    df_scores = pd.DataFrame()
df_scores

## 10. What's next

- **Bigger feature sets** — swap `'small'` for `'medium'` / `'all'` (the exact sizes are printed in step 3; `'all'` is the full set).
- **Auxiliary targets** — train on the other peaks (`target_k2_20`, `target_lhotse_20`, ...) and ensemble them; in EIQ they are all populated, so no NaN handling is needed. Remember `manaslu` and `nanga_parbat` are inverses (corr ≈ −1.0) — keep only one.
- **Feature neutralization** — reduce feature exposure to lift AIMC.
- **Docs:** https://docs.everesteer.ai